An asynchronous function (or coroutine) in Python is a function that can pause its execution while waiting for an external operation (like a network request or disk read) to finish, allowing the program to handle other tasks in the meantime.

#### Keywords

- async def: Declares a function as a co-routine. Calling an async def function does not immediately run its body; it returns a coroutine object.

- await: Pauses the execution of the coroutine until the awaited awaitable (another coroutine, Task, or Future) completes. It can only be used inside an async def function.

- Event Loop: The scheduler that coordinates execution. It runs tasks, manages I/O events, and switches between coroutines whenever an await yields control.

- asyncio.run(): The standard entry point that starts the event loop, runs the passed root coroutine until completion, and closes the loop.

- asyncio.gather(): Runs multiple coroutines concurrently and collects their results in order.

In [ ]:
#Running multiple time consuming tasks in a loop
import time
def fetch_data(task_id:int, delay:int):
    print(f"Task {task_id} started.. waiting {delay} sec..")
    time.sleep(delay)
    print(f"task {task_id} finished!")

def main():
    start_time = time.time()
    fetch_data(1,2)
    fetch_data(2,3)
    fetch_data(3,1)
    elapsed_time = time.time()-start_time
    print(f"Total time taken:{elapsed_time}")
main()

In [ ]:
#Running multiple tasks in parallel using async -- await
import asyncio
import time
async def fetch_data_async(task_id:int, delay:int):
    print(f"Task {task_id} started.. waiting {delay} sec..")
    await asyncio.sleep(delay)
    print(f"Task {task_id}: Finished!")
async def main():
    start_time =time.time()
    results = await asyncio.gather(
        fetch_data_async(1,2),
        fetch_data_async(2,3),
        fetch_data_async(3,1)
    )
    print(f"Results:{results}")
    print(f"Total time taken: {time.time()-start_time:.2f} sec")

await main()

## Async function use with LLM prompting for better performance

#### prompt running as a simple loop without using async function

In [ ]:
#sending multiple prompts in a loop to LLM
import time
from google import genai
from dotenv import load_dotenv
load_dotenv()
client = genai.Client()
prompt = [
    "Explain gravity in one sentence.",
    "Give me a haiku about AI",
    "Name three primary colors",
    "What is the capital of Japan"
]
def get_response(prompt:str)->str:
    response = client.models.generate_content(
        model="gemini-3.7-flash",
        contents=prompt
    )
    return response.text
start_time=time.time()
result=[]
for idx, p in enumerate(prompt,start=1):
    print(f"[{idx}/{len(prompt)}] sending: '{p}'")
    res = get_response(p)
    result.append(res)
    print(f"Response:\n{res.strip()}\n{'-'*40}")
elapsed_time =time.time() - start_time
print(f"Total time taken:{elapsed_time:.2f} sec")
#11.02 sec

### Use Async for multiple LLM prompts in parallel

In [22]:
#sending Multiple prompts to LLM in parallel using async function

import time
import asyncio
from google import genai

client = genai.Client()

prompts = [
    "Explain gravity in one sentence.",
    "Give me a haiku about AI.",
    "Name three primary colors.",
    "What is the capital of Japan?",
]

async def fetch_async(prompt_idx: int, prompt: str) -> str:
    print(f"[{prompt_idx}/{len(prompts)}] Sending: '{prompt}'")
    
    response = await client.aio.models.generate_content(
        model="gemini-3.7-flash",
        contents=prompt,
    )
    
    print(f"[{prompt_idx}/{len(prompts)}] Received response!")
    return response.text

# Measure execution directly in the notebook cell
start_time = time.perf_counter()

# Top-level await is natively supported in Jupyter
tasks = [fetch_async(idx, p) for idx, p in enumerate(prompts, start=1)]
results = await asyncio.gather(*tasks)

elapsed = time.perf_counter() - start_time

print("\n" + "=" * 40 + "\nAll Responses Received:\n" + "=" * 40)
for p, res in zip(prompts, results):
    print(f"Prompt: {p}")
    print(f"Response:\n{res.strip()}\n{'-'*40}")

print(f"Total Asynchronous Time: {elapsed:.2f} seconds")

Direct use of automatic function calling (AFC) in AsyncModels.generate_content is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message. Similarly, direct use of AFC in AsyncModels.generate_content_stream is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message_stream.


[1/4] Sending: 'Explain gravity in one sentence.'
[2/4] Sending: 'Give me a haiku about AI.'
[3/4] Sending: 'Name three primary colors.'
[4/4] Sending: 'What is the capital of Japan?'
[4/4] Received response!
[1/4] Received response!
[3/4] Received response!
[2/4] Received response!

All Responses Received:
Prompt: Explain gravity in one sentence.
Response:
Gravity is the fundamental force of nature that pulls objects with mass toward one another, keeping our feet on the ground and the planets in orbit around the Sun.
----------------------------------------
Prompt: Give me a haiku about AI.
Response:
Lines of silent code,  
Learning from the human spark,  
A new mind awakes.
----------------------------------------
Prompt: Name three primary colors.
Response:
The three traditional primary colors (in art and pigment) are:

1. **Red**
2. **Yellow**
3. **Blue**

*(Note: In the context of light/digital screens, the additive primary colors are **Red**, **Green**, and **Blue**.)*
----------